In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import t
from scipy.stats import ttest_rel

from polylex_chatbot.config import EVALUATION_RESULTS_PATH

In [2]:
def compare_collections(collection_names, corpus_name, configuration_name, ref_collection_name, other_collection_names, metric):
    mrr_results = pd.DataFrame()

    for collection_name in collection_names:
        configuration_path = os.path.join(
            EVALUATION_RESULTS_PATH,
            corpus_name,
            collection_name,
            configuration_name,
            "df_scores_ordered.csv"
        )
        config_results_df = pd.read_csv(configuration_path)
        mrr_results[collection_name] = config_results_df[metric].values

    reference = ref_collection_name
    others = other_collection_names
    ttest_results_list = []
    for other in others:
        ref_values = mrr_results[reference]
        other_values = mrr_results[other]
        diff = ref_values - other_values
        t_stat, p_value = ttest_rel(ref_values, other_values)
        n = len(diff)
        dfree = n - 1
        mean_diff = diff.mean()
        std_diff = diff.std(ddof=1)
        se_diff = std_diff / np.sqrt(n)
        ci_low, ci_high = t.interval(
            confidence=0.95,
            df=dfree,
            loc=mean_diff,
            scale=se_diff
        )
        ttest_results_list.append({
            "metric": metric,
            "comparison": f"{reference} vs {other}",
            "n": n,
            "mean_ref": ref_values.mean(),
            "mean_other": other_values.mean(),
            "mean_diff": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "t": t_stat,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

    ttest_results_df = pd.DataFrame(ttest_results_list)

    return ttest_results_df

In [3]:
def compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric):
    ref_name = configuration_names[0]
    other_name = configuration_names[1]

    ttest_results_list = []

    for collection_name in collection_names:
        ref_path = os.path.join(
            EVALUATION_RESULTS_PATH,
            corpus_name,
            collection_name,
            ref_name,
            "df_scores_ordered.csv"
        )

        other_path = os.path.join(
            EVALUATION_RESULTS_PATH,
            corpus_name,
            collection_name,
            other_name,
            "df_scores_ordered.csv"
        )

        ref_df = pd.read_csv(ref_path)
        other_df = pd.read_csv(other_path)

        ref_values = ref_df[metric].values
        other_values = other_df[metric].values

        diff = ref_values - other_values
        t_stat, p_value = ttest_rel(ref_values, other_values)
        n = len(diff)
        dfree = n - 1
        mean_diff = diff.mean()
        std_diff = diff.std(ddof=1)
        se_diff = std_diff / np.sqrt(n)
        ci_low, ci_high = t.interval(
            confidence=0.95,
            df=dfree,
            loc=mean_diff,
            scale=se_diff
        )
        ttest_results_list.append({
            "metric": metric,
            "collection": collection_name,
            "comparison": f"{ref_name} vs {other_name}",
            "n": n,
            "mean_ref": ref_values.mean(),
            "mean_other": other_values.mean(),
            "mean_diff": mean_diff,
            "ci95_low": ci_low,
            "ci95_high": ci_high,
            "t": t_stat,
            "df": dfree,
            "p_value": p_value,
            "significant": p_value < 0.05
        })

    ttest_results_df = pd.DataFrame(ttest_results_list)

    return ttest_results_df

# Comparaisons retrieval

## Ttest pour savoir si configuration_e significativement meilleure sur dense + no reranker par rapport aux autres collections -> pas le cas

```
PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_a" --run-name="configuration_a_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:13:58.585693Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_c" --run-name="configuration_c_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:25.599367Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_d" --run-name="configuration_d_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:32.555443Z" --name-dir="retrieval_dense_no_reranker"

PYTHONPATH="$PWD/src" python scripts/analyze_run.py --env-path="./envs/.env.configuration_f" --run-name="configuration_f_mistralai/Mistral-Small-3.2-24B-Instruct-2506 - 2026-07-04T11:19:44.643530Z" --name-dir="retrieval_dense_no_reranker"
```

In [4]:
collection_names = ["configuration_a", "configuration_b", "configuration_c", "configuration_d", "configuration_e", "configuration_f"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_name = "retrieval_dense_no_reranker"
ref_collection_name = "configuration_e"
other_collection_names = ["configuration_a", "configuration_b", "configuration_c", "configuration_d", "configuration_f"]
metric = "mrr_doc"

retrieval_ttest_results_dense_no_reranked_df = compare_collections(collection_names, corpus_name, configuration_name, ref_collection_name, other_collection_names, metric)

retrieval_ttest_results_dense_no_reranked_df

,metric,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,p_value,significant
0,mrr_doc,configuration_e vs configuration_a,13,0.026426,0.000000,0.026426,-0.009306,0.062158,1.611386,0.133069,False
1,mrr_doc,configuration_e vs configuration_b,13,0.026426,0.014685,0.011741,-0.029900,0.053382,0.614327,0.550469,False
2,mrr_doc,configuration_e vs configuration_c,13,0.026426,0.000000,0.026426,-0.009306,0.062158,1.611386,0.133069,False
3,mrr_doc,configuration_e vs configuration_d,13,0.026426,0.013889,0.012537,-0.032853,0.057927,0.601817,0.558495,False
4,mrr_doc,configuration_e vs configuration_f,13,0.026426,0.015385,0.011042,-0.041132,0.063215,0.461110,0.652965,False


## Ttest pour savoir si dense VS hybrid pour collection_b et collection_e est significatif -> c'est le cas

In [5]:
collection_names = ["configuration_b", "configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_dense_no_reranker", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_dense_vs_hybrid_no_reranked_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_dense_vs_hybrid_no_reranked_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_dense_no_reranker vs retrieval_hybri...,13,0.014685,0.384615,-0.369930,-0.605484,-0.134376,-3.421760,12,0.005062,True
1,mrr_doc,configuration_e,retrieval_dense_no_reranker vs retrieval_hybri...,13,0.026426,0.356410,-0.329984,-0.549645,-0.110323,-3.273108,12,0.006665,True


## Ttest pour savoir si sparse VS hybrid pour collection_b et collection_e est significatif -> pas le cas (mais sparse quand meme meilleur que hybrid selon certains recall@k observes)

In [6]:
collection_names = ["configuration_b", "configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_sparse_no_reranker", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_sparse_vs_hybrid_no_reranked_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_sparse_vs_hybrid_no_reranked_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_sparse_no_reranker vs retrieval_hybr...,13,0.546154,0.384615,0.161538,-0.173821,0.496898,1.049508,12,0.314622,False
1,mrr_doc,configuration_e,retrieval_sparse_no_reranker vs retrieval_hybr...,13,0.546154,0.356410,0.189744,-0.178306,0.557793,1.123261,12,0.283297,False


## Ttest pour savoir si l'un des rerankers est significativement meilleur pour collection_b et collection_e -> pas le cas (sur dense et hybrid, sparse on sait pas)

FIXME : besoin de run sparse_reranker_qwen sur b et e ?

In [7]:
collection_names = ["configuration_b", "configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_dense_reranker_bge", "retrieval_dense_reranker_qwen"]
metric = "mrr_doc"

retrieval_ttest_results_dense_rerankers_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_dense_rerankers_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.097436,0.104701,-0.007265,-0.261177,0.246647,-0.062341,12,0.951318,False
1,mrr_doc,configuration_e,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.086538,0.042308,0.044231,-0.140194,0.228655,0.522547,12,0.610801,False


In [8]:
collection_names = ["configuration_b", "configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_hybrid_reranker_bge", "retrieval_hybrid_reranker_qwen"]
metric = "mrr_doc"

retrieval_ttest_results_hybrid_rerankers_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_hybrid_rerankers_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_b,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.635294,0.560562,0.074732,-0.230528,0.379993,0.533407,12,0.603491,False
1,mrr_doc,configuration_e,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.641758,0.511538,0.130220,-0.232340,0.492780,0.782559,12,0.449054,False


## Ttest pour savoir si le reranker bge ameliore significativement sparse, dense et hybrid pour collection_e -> malheureusement pas le cas

In [9]:
collection_names = ["configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_sparse_reranker_bge", "retrieval_sparse_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_sparses_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_sparses_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_sparse_reranker_bge vs retrieval_spa...,13,0.700855,0.546154,0.154701,-0.211222,0.520624,0.921135,12,0.375128,False


In [10]:
collection_names = ["configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_dense_reranker_bge", "retrieval_dense_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_denses_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_denses_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_dense_reranker_bge vs retrieval_dens...,13,0.086538,0.026426,0.060112,-0.114901,0.235125,0.748364,12,0.468655,False


In [11]:
collection_names = ["configuration_e"]
corpus_name = "20260702_configurations_comparison_corpus"
configuration_names = ["retrieval_hybrid_reranker_bge", "retrieval_hybrid_no_reranker"]
metric = "mrr_doc"

retrieval_ttest_results_hybrids_df = compare_configurations_per_collections(collection_names, corpus_name, configuration_names, metric)

retrieval_ttest_results_hybrids_df

,metric,collection,comparison,n,mean_ref,mean_other,mean_diff,ci95_low,ci95_high,t,df,p_value,significant
0,mrr_doc,configuration_e,retrieval_hybrid_reranker_bge vs retrieval_hyb...,13,0.641758,0.35641,0.285348,-0.116503,0.687199,1.54714,12,0.147786,False


## Configuration finale pour les hyperparametres du retrieval

- taille des chunks de 2'000 caracteres
- qwen comme modele d'embeddings
- mode de recuperation hybride
- bge comme reranker

In [ ]:
results_best_retrieval = pd.read_csv(best_retrieval_results_path)

In [12]:
# TODO

## Configuration finale pour les hyperparametres de generation

- prompts:
    - PROMPT_TEMPLATE_FR="Réponds à la question en utilisant uniquement le contexte fourni.\n\nContexte:\n{context_text}\n\nQuestion:\n{query}\n\nRéponse:"
    - PROMPT_TEMPLATE_EN="Answer the question using only the provided context.\n\nContext:\n{context_text}\n\nQuestion:\n{query}\n\nAnswer:"
- mistral comme LLM
- contexte contenant soit des documents, des chunks ou un mix des deux selon apparitions repetees de chunks venant d'un meme document et score superieur a un seuil de 0.2 pour chunks uniques
- mistral comme LLM-as-Judge (et qwen pour les embeddings)

In [27]:
results_best_retrieval["mrr_doc"]

0     0.200000
1     1.000000
2     0.000000
3     1.000000
4     1.000000
5     0.142857
6     1.000000
7     0.000000
8     1.000000
9     1.000000
10    1.000000
11    0.500000
12    0.500000
Name: mrr_doc, dtype: float64

In [9]:
print(EVALUATION_RESULTS_PATH / "20260702_configurations_comparison_corpus" )

/home/saskya/dev/tb/polylex-chatbot/evaluations/20260702_configurations_comparison_corpus


# Comparaisons generation